Project: /mediapipe/_project.yaml
Book: /mediapipe/_book.yaml

<link rel="stylesheet" href="/mediapipe/site.css">

# Hand gesture recognition model customization guide

<table align="left" class="buttons">
  <td>
    <a href="https://colab.research.google.com/github/googlesamples/mediapipe/blob/main/examples/customization/gesture_recognizer.ipynb" target="_blank">
      <img src="https://developers.google.com/static/mediapipe/solutions/customization/colab-logo-32px_1920.png" alt="Colab logo"> Run in Colab
    </a>
  </td>

  <td>
    <a href="https://github.com/googlesamples/mediapipe/blob/main/examples/customization/gesture_recognizer.ipynb" target="_blank">
      <img src="https://developers.google.com/static/mediapipe/solutions/customization/github-logo-32px_1920.png" alt="GitHub logo">
      View on GitHub
    </a>
  </td>
</table>

In [6]:
#@title License information
# Copyright 2023 The MediaPipe Authors.
# Licensed under the Apache License, Version 2.0 (the "License");
#
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

The MediaPipe Model Maker package is a low-code solution for customizing on-device machine learning (ML) Models.

This notebook shows the end-to-end process of customizing a gesture recognizer model for recognizing some common hand gestures in the [HaGRID](https://www.kaggle.com/datasets/innominate817/hagrid-sample-30k-384p) dataset.

## Prerequisites

Install the MediaPipe Model Maker package.

In [7]:
# !pip install --upgrade pip
!pip install mediapipe-model-maker

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


Import the required libraries.

In [8]:
import os
import tensorflow as tf
assert tf.__version__.startswith('2')

from mediapipe_model_maker import gesture_recognizer

import matplotlib.pyplot as plt

## Simple End-to-End Example

This end-to-end example uses Model Maker to customize a model for on-device gesture recognition.

### Get the dataset

The dataset for gesture recognition in model maker requires the following format: `<dataset_path>/<label_name>/<img_name>.*`. In addition, one of the label names (`label_names`) must be `none`. The `none` label represents any gesture that isn't classified as one of the other gestures.

This example uses a rock paper scissors dataset sample which is downloaded from GCS.

In [13]:
# !wget https://storage.googleapis.com/mediapipe-tasks/gesture_recognizer/rps_data_sample.zip
# !unzip rps_data_sample.zip
dataset_path = "/media/isr-lab-4/Main/nikita_ws/hoverAI_2_UIST/HoverAI_2/gesture_real_time_control/cropped_data"#"/media/imit-learn/ISR_2T2/HoverAI_2/gesture_real_time_control/data/cropped"
export_dir = "/media/isr-lab-4/Main/nikita_ws/hoverAI_2_UIST/HoverAI_2/gesture_real_time_control/models"

In [15]:
os.path.exists(dataset_path)


True

Verify the rock paper scissors dataset by printing the labels. There should be 4 gesture labels, with one of them being the `none` gesture.

In [16]:
print(dataset_path)
labels = []
for i in os.listdir(dataset_path):
  if os.path.isdir(os.path.join(dataset_path, i)):
    labels.append(i)
print(labels)

/media/isr-lab-4/Main/nikita_ws/hoverAI_2_UIST/HoverAI_2/gesture_real_time_control/cropped_data
['none', 'rock', 'palm']


To better understand the dataset, plot a couple of example images for each gesture.

### Run the example
The workflow consists of 4 steps which have been separated into their own code blocks.

**Load the dataset**

Load the dataset located at `dataset_path` by using the `Dataset.from_folder` method. When loading the dataset, run the pre-packaged hand detection model from MediaPipe Hands to detect the hand landmarks from the images. Any images without detected hands are ommitted from the dataset. The resulting dataset will contain the extracted hand landmark positions from each image, rather than images themselves.

The `HandDataPreprocessingParams` class contains two configurable options for the data loading process:
* `shuffle`: A boolean controlling whether to shuffle the dataset. Defaults to true.
* `min_detection_confidence`: A float between 0 and 1 controlling the confidence threshold for hand detection.

Split the dataset: 80% for training, 10% for validation, and 10% for testing.

In [17]:
data = gesture_recognizer.Dataset.from_folder(
    dirname=dataset_path,
    hparams=gesture_recognizer.HandDataPreprocessingParams()
)
train_data, rest_data = data.split(0.8)
validation_data, test_data = rest_data.split(0.5)

INFO:tensorflow:Loading image /media/isr-lab-4/Main/nikita_ws/hoverAI_2_UIST/HoverAI_2/gesture_real_time_control/cropped_data/palm/2026-03-23 19-38-54_93485973.JPG
INFO:tensorflow:Loading image /media/isr-lab-4/Main/nikita_ws/hoverAI_2_UIST/HoverAI_2/gesture_real_time_control/cropped_data/palm/PXL_20260322_215833262.jpg
INFO:tensorflow:Loading image /media/isr-lab-4/Main/nikita_ws/hoverAI_2_UIST/HoverAI_2/gesture_real_time_control/cropped_data/palm/2026-03-23 20-22-07_1774286749_91442813.JPG
INFO:tensorflow:Loading image /media/isr-lab-4/Main/nikita_ws/hoverAI_2_UIST/HoverAI_2/gesture_real_time_control/cropped_data/none/1376.jpg
INFO:tensorflow:Loading image /media/isr-lab-4/Main/nikita_ws/hoverAI_2_UIST/HoverAI_2/gesture_real_time_control/cropped_data/rock/2026-03-23 20-11-41.JPG
INFO:tensorflow:Loading image /media/isr-lab-4/Main/nikita_ws/hoverAI_2_UIST/HoverAI_2/gesture_real_time_control/cropped_data/palm/2026-03-23 20-31-29.JPG
INFO:tensorflow:Loading image /media/isr-lab-4/Main/n

I0000 00:00:1774361950.069349 1727818 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1774361950.155389 1729478 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 580.126.20), renderer: NVIDIA GeForce RTX 4090/PCIe/SSE2
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1774361950.166608 1729481 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1774361950.177738 1729487 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
/home/isr-lab-4/miniconda3/envs/hover_ai_2/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype

INFO:tensorflow:Loading image /media/isr-lab-4/Main/nikita_ws/hoverAI_2_UIST/HoverAI_2/gesture_real_time_control/cropped_data/palm/2026-03-23 20-31-23_1774287156.JPG
INFO:tensorflow:Loading image /media/isr-lab-4/Main/nikita_ws/hoverAI_2_UIST/HoverAI_2/gesture_real_time_control/cropped_data/rock/2026-03-23 19-22-34.JPG
INFO:tensorflow:Loading image /media/isr-lab-4/Main/nikita_ws/hoverAI_2_UIST/HoverAI_2/gesture_real_time_control/cropped_data/rock/2026-03-23 19-39-45.JPG
INFO:tensorflow:Loading image /media/isr-lab-4/Main/nikita_ws/hoverAI_2_UIST/HoverAI_2/gesture_real_time_control/cropped_data/palm/PXL_20260322_215110281.jpg
INFO:tensorflow:Loading image /media/isr-lab-4/Main/nikita_ws/hoverAI_2_UIST/HoverAI_2/gesture_real_time_control/cropped_data/palm/2026-03-23 20-22-02.JPG
INFO:tensorflow:Loading image /media/isr-lab-4/Main/nikita_ws/hoverAI_2_UIST/HoverAI_2/gesture_real_time_control/cropped_data/rock/2026-03-23 20-22-32_1774286701.JPG
INFO:tensorflow:Loading image /media/isr-lab-

2026-03-24 17:19:52.658521: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-03-24 17:19:52.749779: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2256] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


INFO:tensorflow:Load valid hands with size: 1144, num_label: 3, labels: none,palm,rock.


INFO:tensorflow:Load valid hands with size: 1144, num_label: 3, labels: none,palm,rock.


**Train the model**

Train the custom gesture recognizer by using the create method and passing in the training data, validation data, model options, and hyperparameters. For more information on model options and hyperparameters, see the [Hyperparameters](#hyperparameters) section below.

In [18]:
hparams = gesture_recognizer.HParams(export_dir=export_dir)
options = gesture_recognizer.GestureRecognizerOptions(hparams=hparams)
model = gesture_recognizer.GestureRecognizer.create(
    train_data=train_data,
    validation_data=validation_data,
    options=options
)




Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 hand_embedding (InputLayer  [(None, 128)]             0         
 )                                                               
                                                                 
 batch_normalization (Batch  (None, 128)               512       
 Normalization)                                                  
                                                                 
 re_lu (ReLU)                (None, 128)               0         
                                                                 
 dropout (Dropout)           (None, 128)               0         
                                                                 
 custom_gesture_recognizer_  (None, 3)                 387       
 out (Dense)                                                     
                                                             

INFO:tensorflow:Training the models...


Epoch 1/10
457/457 [==============================] - 1s 1ms/step - loss: 0.3948 - categorical_accuracy: 0.5875 - val_loss: 0.2947 - val_categorical_accuracy: 0.8158 - lr: 0.0010
Epoch 2/10
457/457 [==============================] - 0s 894us/step - loss: 0.3050 - categorical_accuracy: 0.6652 - val_loss: 0.2034 - val_categorical_accuracy: 0.8509 - lr: 9.9000e-04
Epoch 3/10
457/457 [==============================] - 0s 832us/step - loss: 0.2751 - categorical_accuracy: 0.6805 - val_loss: 0.1817 - val_categorical_accuracy: 0.8860 - lr: 9.8010e-04
Epoch 4/10
457/457 [==============================] - 0s 889us/step - loss: 0.2619 - categorical_accuracy: 0.6937 - val_loss: 0.1805 - val_categorical_accuracy: 0.8772 - lr: 9.7030e-04
Epoch 5/10
457/457 [==============================] - 0s 867us/step - loss: 0.2473 - categorical_accuracy: 0.7035 - val_loss: 0.1771 - val_categorical_accuracy: 0.8772 - lr: 9.6060e-04
Epoch 6/10
457/457 [==============================] - 0s 840us/step - loss: 0.244

**Evaluate the model performance**

After training the model, evaluate it on a test dataset and print the loss and accuracy metrics.

In [19]:
loss, acc = model.evaluate(test_data, batch_size=1)
print(f"Test loss:{loss}, Test accuracy:{acc}")

115/115 [==============================] - 0s 329us/step - loss: 0.1860 - categorical_accuracy: 0.8783
Test loss:0.18602725863456726, Test accuracy:0.8782608509063721


**Export to Tensorflow Lite Model**

After creating the model, convert and export it to a Tensorflow Lite model format for later use on an on-device application. The export also includes model metadata, which includes the label file.

In [20]:
model.export_model()
# !ls exported_model

Using existing files at /tmp/model_maker/gesture_recognizer/palm_detection_full.tflite
Using existing files at /tmp/model_maker/gesture_recognizer/hand_landmark_full.tflite
INFO:tensorflow:Assets written to: /tmp/tmpor4lpc0z/saved_model/assets


INFO:tensorflow:Assets written to: /tmp/tmpor4lpc0z/saved_model/assets
2026-03-24 17:20:01.620306: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-03-24 17:20:01.620321: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-03-24 17:20:01.620500: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpor4lpc0z/saved_model
2026-03-24 17:20:01.620897: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-03-24 17:20:01.620902: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpor4lpc0z/saved_model
2026-03-24 17:20:01.621901: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:388] MLIR V1 optimization pass is not enabled
2026-03-24 17:20:01.622198: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-03-24 17:20:01.633824: I tensorflow/cc/saved_model/